# 01 FTH — image series / hysteresis loop

Apply the calibration from the original `01_FTH.ipynb` (and, when available,
`03_define_supportmask.ipynb`) to many `+`/`-` image pairs. The calibrated
image-13 center, pixel mask, smooth beamstop mask, Ewald setting, FTH propagation,
global phase, and ROI are reused unchanged.

Run the original notebooks on one representative pair first. Edit only the
configuration cell below for a new series.

In [ ]:
# Configure Qt before importing pyplot or any other GUI-aware library.
import os
import sys
from os.path import join

BASEFOLDER = os.path.abspath(os.getcwd())
LIBRARY_FOLDER = join(BASEFOLDER, "library")
if LIBRARY_FOLDER not in sys.path:
    sys.path.insert(0, LIBRARY_FOLDER)

from notebook_setup import configure_matplotlib_qt
MATPLOTLIB_BACKEND = configure_matplotlib_qt()
print("Matplotlib backend:", MATPLOTLIB_BACKEND)

import os, sys
from os.path import join

import numpy as np
import matplotlib.pyplot as plt

BASEFOLDER = os.path.abspath(os.getcwd())
sys.path.insert(0, join(BASEFOLDER, "library"))

import importlib
import CCI_core as cci
from interactive import cimshow
import fthcore as fth
import fth_phase_workflow as wf
wf = importlib.reload(wf)  # Refresh helpers in an existing kernel.
from data_loading import SextantsNexusLoader, image_ids, load_average

try:
    import Phase_Retrieval as PhR
except Exception:
    PhR = None

print("Base folder:", BASEFOLDER)

## Configuration

In [ ]:
USER = "rb"
RAW_FOLDER = "/nfs/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/"
# Explicitly choose the positive image whose 01_FTH result is the reference.
REFERENCE_IMAGE_ID = 95
REFERENCE_H5 = join(
    BASEFOLDER, "processed", "Logs",
    f"data_recon_ImId_{REFERENCE_IMAGE_ID:04d}_{USER}.hdf5",
)
SERIES_H5 = join(BASEFOLDER, "processed", "Logs", f"data_recon_series_{USER}.hdf5")

# Define parallel series entries. Each entry may be one ID or a list to average.
PLUS_IMAGE_IDS = list(np.arange(53, 96,2))       # Example: [13, 15, 17, 19]
MINUS_IMAGE_IDS = list(np.arange(54, 97,2))      # Example: [14, 16, 18, 20]
SERIES_POINTS = None        # Example fields: [-100, -50, 0, 50]

if len(PLUS_IMAGE_IDS) != len(MINUS_IMAGE_IDS):
    raise ValueError("PLUS_IMAGE_IDS and MINUS_IMAGE_IDS must have equal length")
if not PLUS_IMAGE_IDS:
    raise ValueError("Define at least one + / - image pair")
if SERIES_POINTS is None:
    SERIES_POINTS = list(range(len(PLUS_IMAGE_IDS)))
if len(SERIES_POINTS) != len(PLUS_IMAGE_IDS):
    raise ValueError("SERIES_POINTS must have one value per image pair")
SERIES = [
    {"point": point, "+": plus_id, "-": minus_id}
    for point, plus_id, minus_id in zip(
        SERIES_POINTS, PLUS_IMAGE_IDS, MINUS_IMAGE_IDS
    )
]

# Optional dark IDs can be one global pair or overridden inside any SERIES row.
DARK_IDS = {"+": None, "-": None}

# True applies the factor/offset measured in the calibration notebook unchanged.
# Set False only when each pair should be fitted independently for intensity drift.
REUSE_CALIBRATION_SCALING = True
# None means: use the values saved by 01_FTH for REFERENCE_IMAGE_ID.
# Enter lists here only when you intentionally want to override that reference.
CENTER_OVERRIDE = None  # Example: [1005, 1035]
ROI_OVERRIDE = None     # Example: [597, 785, 526, 719]
GIF_FRAME_DURATION_MS = 250
OVERWRITE = True

## Load and validate the calibration

In [ ]:
calibration = wf.load_data_dict(REFERENCE_H5)
labels = ["+", "-"]
if any(label not in calibration["holo"] for label in labels):
    raise ValueError(f"Calibration must contain + and - modes; found {list(calibration['holo'])}")

saved_reference_id = int(image_ids(calibration["holo"]["+"]["id"])[0])
if saved_reference_id != REFERENCE_IMAGE_ID:
    raise ValueError(
        f"Reference file contains positive image {saved_reference_id}, "
        f"not requested image {REFERENCE_IMAGE_ID}"
    )
stored_center = np.asarray(calibration["center"], dtype=float)
center = stored_center if CENTER_OVERRIDE is None else np.asarray(CENTER_OVERRIDE, dtype=float)
experimental_setup = dict(calibration["experimental_setup"])
mask_pixel = np.asarray(calibration["mask_pixel"], dtype=np.uint8)
beamstop_recipe = dict(
    calibration.get("mask_beamstop_smooth_recipe")
    or calibration.get("mask_pixel_smooth_recipe", {"radius": 35, "order": 4})
)
mask_beamstop_smooth = wf.butterworth_disk_mask(
    mask_pixel.shape, beamstop_recipe["radius"], beamstop_recipe["order"]
)
mask_pixel_fth_recipe = dict(
    calibration.get("mask_pixel_fth_recipe", {"dilation_pixels": 3, "sigma": 3})
)
mask_pixel_fth = wf.smooth_binary_mask(
    mask_pixel.astype(float),
    mask_pixel_fth_recipe["dilation_pixels"],
    mask_pixel_fth_recipe["sigma"],
)
mask_multiplier = (1 - mask_beamstop_smooth) * (1 - mask_pixel_fth)
focus_fth = dict(calibration.get("focus_fth", {}))
prop_dist = float(focus_fth.get("prop_dist", 0))  # micrometres
prop_dist_unit = focus_fth.get("prop_dist_unit", "um")
if prop_dist_unit not in ("um", "µm"):
    raise ValueError(f"Expected FTH propagation in micrometres, got {prop_dist_unit!r}")
phase = float(focus_fth.get("phase", 0))
dx = float(focus_fth.get("dx", 0))
dy = float(focus_fth.get("dy", 0))
stored_roi = focus_fth.get("roi")
if stored_roi is None and ROI_OVERRIDE is None:
    raise ValueError("The reference has no saved ROI; finish 01_FTH.ipynb first")
roi = list(stored_roi if ROI_OVERRIDE is None else ROI_OVERRIDE)
roi_s = np.s_[roi[0] : roi[1], roi[2] : roi[3]]
focus_fth["roi"] = roi
project_ewalds_sphere = bool(calibration.get("project_ewalds_sphere", False))
ewald_method = calibration.get("ewald_method", "cubic")

if project_ewalds_sphere and PhR is None:
    raise RuntimeError("Calibration requests Ewald projection, but Phase_Retrieval could not import")
if tuple(mask_pixel.shape) != tuple(calibration["holo"]["+"]["image_c"].shape):
    raise ValueError("Calibration mask and centered hologram shapes differ")

print("Reference image:", REFERENCE_IMAGE_ID)
print("Reference HDF5:", REFERENCE_H5)
print("Center:", center, "mask pixels:", int(mask_pixel.sum()), "ROI:", roi)
print("FTH focus: prop_dist=", prop_dist, "phase=", phase)

## Process every image pair

In [ ]:
loader = SextantsNexusLoader(RAW_FOLDER)
series_data = {
    "workflow": "FTH_series",
    "reference_image_id": REFERENCE_IMAGE_ID,
    "calibration_file": REFERENCE_H5,
    "experimental_setup": experimental_setup,
    "positive_label": "+",
    "reference_label": "-",
    "hologram_labels": labels,
    "center": center,
    "mask_pixel": mask_pixel,
    "mask_beamstop_smooth_recipe": beamstop_recipe,
    "mask_pixel_fth_recipe": mask_pixel_fth_recipe,
    "focus_fth": {**focus_fth, "prop_dist_unit": "um"},
    "project_ewalds_sphere": project_ewalds_sphere,
    "ewald_method": ewald_method,
    "supportmask": calibration.get("supportmask"),
    "focus_cdi": calibration.get("focus_cdi", {}),
    "phase_retrieval_recipe": calibration.get("phase_retrieval_recipe"),
    "series": [],
}

for index, spec in enumerate(SERIES):
    entry = {
        "index": index,
        "point": spec.get("point", index),
        "holo": {},
    }
    for label in labels:
        requested_ids = image_ids(spec[label])
        image_id = int(requested_ids[0])
        frame = load_average(loader, requested_ids)
        image = np.asarray(frame.image, dtype=float)
        dark_id = spec.get(f"{label}_dark", DARK_IDS[label])
        if dark_id is not None:
            image = image - np.asarray(load_average(loader, dark_id).image, dtype=float)
        centered = wf.center_image(image, center, cci)
        if project_ewalds_sphere:
            centered = PhR.inv_gnomonic(
                centered,
                center=np.asarray(centered.shape) / 2,
                experimental_setup=experimental_setup,
                method=ewald_method,
            )
        if centered.shape != mask_pixel.shape:
            raise ValueError(
                f"Series point {index}, {label}: shape {centered.shape} != calibration {mask_pixel.shape}"
            )
        entry["holo"][label] = {
            "id": spec[label],
            "dark_id": dark_id,
            "source": str(frame.source),
            "exposure": frame.exposure,
            "raw_metadata": dict(frame.metadata),
            "image_c": np.asarray(centered),
        }

    pos_raw = np.asarray(entry["holo"]["+"]["image_c"], dtype=float)
    neg_raw = np.asarray(entry["holo"]["-"]["image_c"], dtype=float)
    if REUSE_CALIBRATION_SCALING:
        factor = float(calibration["factor"])
        offset = float(calibration["offset"])
    else:
        factor, offset = cci.dyn_factor(
            pos_raw * (1 - mask_pixel), neg_raw * (1 - mask_pixel),
            method="correlation", verbose=False, plot=False,
        )
        factor, offset = float(factor), float(offset)

    pos = pos_raw / factor
    neg = neg_raw
    holo = (pos - neg - offset) * mask_multiplier
    # wf.fth_reconstruct accepts micrometres and converts to metres once.
    recon = wf.fth_reconstruct(
        holo, experimental_setup, fth, prop_dist=prop_dist, phase=phase, dx=dx, dy=dy
    )
    entry["factor"] = factor
    entry["offset"] = offset
    entry["hologram"] = holo
    entry["recon"] = recon[roi_s]
    series_data["series"].append(entry)

    image_id = int(image_ids(entry["holo"]["+"]["id"])[0])
    png = join(BASEFOLDER, "processed", f"FTH_series_{index:04d}_ImId_{image_id:04d}_{USER}.png")
    fig, ax = plt.subplots(figsize=(5, 5))
    shown = np.real(entry["recon"])
    vmin, vmax = wf.finite_percentile_limits(shown)
    ax.imshow(shown, vmin=vmin, vmax=vmax, cmap="gray")
    ax.set_title(f"point {entry['point']}: +{spec['+']} / -{spec['-']}")
    ax.set_axis_off()
    fig.savefig(png, bbox_inches="tight", dpi=200)
    plt.close(fig)
    entry["fth_png"] = png

    # Checkpoint after each point so a long acquisition can be resumed/recovered.
    wf.save_data_dict(series_data, SERIES_H5, overwrite=True)
    print(f"[{index + 1}/{len(SERIES)}] saved +{spec['+']} / -{spec['-']}")

print("Series HDF5:", SERIES_H5)

## Inspect the reconstructed loop

In [ ]:
stack = np.stack([entry["recon"] for entry in series_data["series"]])
print("Reconstruction stack:", stack.shape)

columns = min(4, len(series_data["series"]))
rows = int(np.ceil(len(series_data["series"]) / columns))
fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows), squeeze=False)
for ax, entry in zip(axes.flat, series_data["series"]):
    shown = np.real(entry["recon"])
    vmin, vmax = wf.finite_percentile_limits(shown)
    ax.imshow(shown, vmin=vmin, vmax=vmax, cmap="gray")
    ax.set_title(f"point {entry['point']}")
    ax.set_axis_off()
for ax in axes.flat[len(series_data["series"]):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

# Create an animated GIF with one consistently scaled frame per reconstruction.
from PIL import Image

real_stack = np.real(stack)
finite = real_stack[np.isfinite(real_stack)]
if not finite.size:
    raise ValueError("Cannot create GIF: reconstructions contain no finite values")
vmin, vmax = np.percentile(finite, (1, 99))
if vmax <= vmin:
    vmax = vmin + 1.0
gif_frames = []
for reconstruction in real_stack:
    normalized = np.clip((reconstruction - vmin) / (vmax - vmin), 0, 1)
    frame = np.nan_to_num(normalized, nan=0, posinf=1, neginf=0)
    gif_frames.append(Image.fromarray(np.uint8(np.rint(frame * 255))))

GIF_PATH = join(BASEFOLDER, "processed", f"FTH_series_{USER}.gif")
gif_frames[0].save(
    GIF_PATH,
    save_all=True,
    append_images=gif_frames[1:],
    duration=GIF_FRAME_DURATION_MS,
    loop=0,
)
series_data["fth_gif"] = GIF_PATH
series_data["gif_frame_duration_ms"] = GIF_FRAME_DURATION_MS
wf.save_data_dict(series_data, SERIES_H5, overwrite=True)
print("Saved GIF:", GIF_PATH)

In [ ]:
# Acquisition ID summary
print("im_ids:", [{label: item["holo"][label].get("id") for label in item["holo"]} for item in series_data["series"]])
print("dark_ids:", [{label: item["holo"][label].get("dark_id") for label in item["holo"]} for item in series_data["series"]])